In [4]:
import json
from pathlib import Path

results_path = Path("wandb_compute_exps/results.json")

with results_path.open("r") as file:
    results = json.load(file)

In [5]:
list(results.keys())

['knn',
 'linear',
 'dnnc',
 'rerank',
 'sklearn',
 'bert',
 'lora',
 'ptuning',
 'system']

In [7]:
_ = results.pop("system")

In [8]:
for scorer_name, res_config_and_metrics in results.items():
    print(f"scorer_name: {scorer_name} has {len(res_config_and_metrics)} runs")

scorer_name: knn has 5 runs
scorer_name: linear has 1 runs
scorer_name: dnnc has 5 runs
scorer_name: rerank has 10 runs
scorer_name: sklearn has 7 runs
scorer_name: bert has 5 runs
scorer_name: lora has 5 runs
scorer_name: ptuning has 5 runs


In [9]:
results["linear"]

[{'config': {'embedder_config': {'value': {'batch_size': 32,
     'classifier_prompt': None,
     'cluster_prompt': None,
     'default_prompt': None,
     'device': None,
     'freeze': True,
     'model_name': 'mixedbread-ai/mxbai-embed-large-v1',
     'passage_prompt': None,
     'query_prompt': None,
     'similarity_fn_name': 'cosine',
     'sts_prompt': None,
     'tokenizer_config': {'max_length': None,
      'padding': True,
      'truncation': True},
     'trust_remote_code': False,
     'use_cache': True}}},
  'metrics': {'emissions/gpu_power': 15.459576302043818,
   'emissions/emissions': 0.0004282634817614922,
   '_step': 0,
   'scoring_recall': 0.9436427707340618,
   'emissions/cpu_count': 16,
   'emissions/duration': -1746690760.1935968,
   'scoring_precision': 0.9486464853393203,
   'emissions/pue': 1,
   'emissions/energy_consumed': 0.0009710331325834955,
   'emissions/gpu_energy': 0.0003121294163701549,
   'scoring_roc_auc': 0.9993575742642029,
   'emissions/ram_energy

In [24]:
from collections import defaultdict
import numpy as np

metrics_to_aggregate = [
    "emissions/emissions",
    "_runtime",
    "emissions/emissons_rate",
    "emissions/energy_consumed",
    "emissions/gpu_power",
    "emissions/cpu_power",
    "emissions/gpu_energy",
    "emissions/cpu_energy",
    "emissions/ram_energy",
    "emissions/emissions_rate",
    "emissions/energy_consumed",
]

aggregated_metrics = {}
for scorer_name, res_config_and_metrics in results.items():
    print(f"scorer_name: {scorer_name} has {len(res_config_and_metrics)} runs")
    grouped_metrics = defaultdict(list)
    for res in res_config_and_metrics:
        for metric in metrics_to_aggregate:
            metric_val = res["metrics"].get(metric, None)
            if metric_val is not None:
                grouped_metrics[metric].append(metric_val)

    medians = {}
    stds = {}
    for metric_name, metric_values in grouped_metrics.items():
        if metric_values:
            # print(metric_name, metric_values)
            medians[metric_name] = np.median(metric_values)
            stds[metric_name] = np.std(metric_values)

    aggregated_metrics[scorer_name] = {"medians": medians, "stds": stds}

aggregated_metrics


scorer_name: knn has 5 runs
scorer_name: linear has 1 runs
scorer_name: dnnc has 5 runs
scorer_name: rerank has 10 runs
scorer_name: sklearn has 7 runs
scorer_name: bert has 5 runs
scorer_name: lora has 5 runs
scorer_name: ptuning has 5 runs


{'knn': {'medians': {'emissions/emissions': np.float64(8.534494756430144e-06),
   '_runtime': np.float64(1.280654019),
   'emissions/energy_consumed': np.float64(1.935088451685712e-05),
   'emissions/gpu_power': np.float64(67.44074098616595),
   'emissions/cpu_power': np.float64(27.0),
   'emissions/gpu_energy': np.float64(1.4050844574065025e-05),
   'emissions/cpu_energy': np.float64(4.339880399638785e-06),
   'emissions/ram_energy': np.float64(9.04411058785016e-07),
   'emissions/emissions_rate': np.float64(1.2260047910820804e-05)},
  'stds': {'emissions/emissions': np.float64(0.0001725836794878742),
   '_runtime': np.float64(14.085032674620392),
   'emissions/energy_consumed': np.float64(0.0003913116062023407),
   'emissions/gpu_power': np.float64(18.677803847970853),
   'emissions/cpu_power': np.float64(0.0),
   'emissions/gpu_energy': np.float64(0.00026359246974820436),
   'emissions/cpu_energy': np.float64(0.00010567229637723178),
   'emissions/ram_energy': np.float64(2.205180147

In [25]:
import pandas as pd

metric = "emissions/emissions"
rows = [{"scorer_name": scorer_name, metric: contents["medians"][metric]} for scorer_name, contents in aggregated_metrics.items()]
df = pd.DataFrame(rows)
df


TypeError: string indices must be integers, not 'str'